# Week 1 Lab — Train a Simple Model

**Objective:** Feel the ML workflow end-to-end. Don't optimize — just understand the steps.

We'll use the Iris dataset (classic, tiny, built into scikit-learn) to:
1. Load data
2. Split into train/test
3. Train a model
4. Make predictions
5. Measure accuracy

This maps to: Data → Train → Evaluate in the ML lifecycle.

## Step 0 — Install dependencies

Run this once if you haven't already:
```bash
pip install scikit-learn pandas matplotlib
```

In [ ]:
# Step 1 — Load the data
from sklearn.datasets import load_iris
import pandas as pd

# Load built-in Iris dataset
iris = load_iris()

# Put it in a DataFrame so it looks like real tabular data
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print(f"Dataset shape: {df.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")
df.head(10)

### Operator's perspective

In production, this data comes from a pipeline (database, API, streaming source).
If the data source changes schema, goes stale, or has nulls — the model degrades **silently**.
This is why data validation matters (Week 2).

In [ ]:
# Step 2 — Explore the data briefly
# As an operator, you want to know: what does healthy data look like?

print("=== Basic statistics (your baseline for data quality) ===")
print(df.describe())
print(f"\nNull values per column:\n{df.isnull().sum()}")
print(f"\nClass distribution:\n{df['species'].value_counts()}")

In [ ]:
# Step 3 — Split into training and test sets
from sklearn.model_selection import train_test_split

# Features (X) and labels (y)
X = df[iris.feature_names]  # input features
y = df['target']            # what we want to predict

# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print("\n--- Why split? ---")
print("Training data = what the model learns from")
print("Test data = simulates 'production' data the model has never seen")
print("If test accuracy is much lower than training accuracy → overfitting")

In [ ]:
# Step 4 — Train a model
from sklearn.ensemble import RandomForestClassifier

# RandomForest: a solid, general-purpose algorithm. Think of it as 
# "multiple decision trees voting together" — you don't need to go deeper.
model = RandomForestClassifier(n_estimators=100, random_state=42)

# .fit() = training. The model looks at X_train + y_train and learns patterns.
model.fit(X_train, y_train)

print("Model trained successfully.")
print(f"Model type: {type(model).__name__}")
print(f"Number of trees: {model.n_estimators}")
print("\n--- Operator insight ---")
print("Training produces a model artifact (like a build artifact).")
print("In production, this gets versioned, registered, and deployed.")

In [ ]:
# Step 5 — Make predictions
y_pred = model.predict(X_test)

# Compare predictions to actual labels
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Correct': y_test.values == y_pred
})

print("First 10 predictions vs actual:")
print(comparison.head(10))
print(f"\nCorrect: {comparison['Correct'].sum()} / {len(comparison)}")

In [ ]:
# Step 6 — Measure accuracy (evaluate the model)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy:.2%}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- What these metrics mean (quick reference) ---")
print("Accuracy: % of correct predictions overall")
print("Precision: of all predicted 'X', how many were actually 'X'")
print("Recall: of all actual 'X', how many did we catch")
print("F1: harmonic mean of precision & recall (balanced view)")
print("\nYou'll go deeper in Week 3. For now, just notice these exist.")

In [ ]:
# Step 7 — Save the model (this becomes your 'artifact')
import joblib

model_path = "model_v1.pkl"
joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")
print(f"File is a serialized Python object — this is what gets deployed.")
print("\n--- Operator insight ---")
print("In production, this artifact goes to a model registry (like MLflow).")
print("You version it, tag it (staging/production), and roll back if needed.")
print("Exactly like your container image versioning.")

In [ ]:
# Step 8 — Simulate "inference in production"
# Load the saved model and predict on new data

loaded_model = joblib.load(model_path)

# Simulate a new flower measurement coming in (this would be an API request in prod)
new_sample = [[5.1, 3.5, 1.4, 0.2]]  # sepal length, sepal width, petal length, petal width
prediction = loaded_model.predict(new_sample)
confidence = loaded_model.predict_proba(new_sample)

print(f"New input: {new_sample[0]}")
print(f"Predicted class: {prediction[0]} ({iris.target_names[prediction[0]]})")
print(f"Confidence scores: {confidence[0]}")
print(f"  setosa: {confidence[0][0]:.2%}, versicolor: {confidence[0][1]:.2%}, virginica: {confidence[0][2]:.2%}")

print("\n--- This is what happens every time your model serves a request ---")
print("Input comes in → model.predict() → response goes out")
print("If the input data starts looking different from training data → drift")
print("If the model returns confident wrong answers → silent failure")

## Summary — What you just did

| Step | ML term | Ops analogy |
|------|---------|------------|
| Load data | Data ingestion | Pull from data source |
| Explore data | Data validation | Health checks on input |
| Split train/test | Train/test split | Staging vs prod |
| Train model | Training | Build step |
| Evaluate | Evaluation | Test suite |
| Save model | Artifact creation | Build artifact / container image |
| Load & predict | Inference / serving | Running the service in prod |

## Key takeaways for an operator

1. **A model is just a file** that takes input and returns output. It's not magic.
2. **Training produces the artifact**, inference uses it. These are different workloads (batch vs real-time).
3. **Accuracy on test data ≠ accuracy in production** — production data drifts over time.
4. **No error is thrown** when accuracy degrades. This is why you need ML-specific monitoring.
5. **Versioning matters** — you need to roll back models just like you roll back deploys.

## Next steps

- Week 2: We'll add data validation (the "health check" for incoming data)
- Week 3: We'll detect drift (the "silent failure" you now understand)